In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import requests
from tqdm.auto import tqdm

url = "https://huggingface.co/datasets/mohanty/PlantVillage/resolve/main/data.zip"
output = "/content/plantvillage_data.zip"

with requests.get(url, stream=True, timeout=60) as r:
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))

    with open(output, "wb") as f:
        with tqdm(
            total=total or None,
            unit="B",
            unit_scale=True,
            desc="Downloading PlantVillage"
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

print("Download complete:", output)

Download complete: /content/plantvillage_data.zip


In [3]:
import zipfile

zip_path = "/content/plantvillage_data.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Total files:", len(z.namelist()))
    for name in z.namelist()[:20]:
        print(name)

Total files: 163034
raw/
raw/color/
raw/color/Raspberry___healthy/
raw/color/Raspberry___healthy/6c2e049f-38c9-42c5-864f-613e6e12d59e___Mary_HL 6318.JPG
raw/color/Raspberry___healthy/d1387960-14d6-4654-ad5c-b92afa86ff6a___Mary_HL 9262.JPG
raw/color/Raspberry___healthy/fafac0a9-e5ba-420d-9dbe-034942f845b9___Mary_HL 9201.JPG
raw/color/Raspberry___healthy/a37f34f7-022d-461a-8a3d-95f5cd774e35___Mary_HL 9155.JPG
raw/color/Raspberry___healthy/fe1c1683-06cc-49c5-b86a-a41fb36f058b___Mary_HL 6249.JPG
raw/color/Raspberry___healthy/e621e3c3-842a-4489-b931-89f05c254cbe___Mary_HL 9347.JPG
raw/color/Raspberry___healthy/b8a96dea-8442-4855-9fdf-c29c9d4eca5a___Mary_HL 6398.JPG
raw/color/Raspberry___healthy/42428dcc-fefc-46dd-b209-150fbeaf4862___Mary_HL 6397.JPG
raw/color/Raspberry___healthy/076d909d-cfca-47ce-a8d1-479bb77c4bab___Mary_HL 6294.JPG
raw/color/Raspberry___healthy/27ef7a65-771c-4eab-9169-03a5fa121c71___Mary_HL 9195.JPG
raw/color/Raspberry___healthy/895d0a2d-5fda-47ca-af52-5cf4f07d610c___Mary

In [4]:
import zipfile
from pathlib import Path

zip_path = "/content/plantvillage_data.zip"
color_dir = Path("/content/plantvillage/raw/color")
color_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    color_files = [
        name for name in z.namelist()
        if name.startswith("raw/color/")
        and not name.endswith("/")
    ]

    print("Files to extract:", len(color_files))
    z.extractall(
        "/content/plantvillage",
        members=color_files
    )

print("Color dataset extraction complete.")

Files to extract: 54305
Color dataset extraction complete.


In [5]:
from pathlib import Path
from collections import Counter

color_dir = Path("/content/plantvillage/raw/color")
extensions = {".jpg", ".jpeg", ".png", ".bmp"}

class_counts = Counter()

for folder in color_dir.iterdir():
    if folder.is_dir():
        count = sum(
            1 for p in folder.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        )
        if count:
            class_counts[folder.name] = count

print("Number of classes:", len(class_counts))
print("Total images:", sum(class_counts.values()))

for name, count in sorted(class_counts.items()):
    print(f"{name}: {count}")

Number of classes: 38
Total images: 54305
Apple___Apple_scab: 630
Apple___Black_rot: 621
Apple___Cedar_apple_rust: 275
Apple___healthy: 1645
Blueberry___healthy: 1502
Cherry_(including_sour)___Powdery_mildew: 1052
Cherry_(including_sour)___healthy: 854
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513
Corn_(maize)___Common_rust_: 1192
Corn_(maize)___Northern_Leaf_Blight: 985
Corn_(maize)___healthy: 1162
Grape___Black_rot: 1180
Grape___Esca_(Black_Measles): 1383
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1076
Grape___healthy: 423
Orange___Haunglongbing_(Citrus_greening): 5507
Peach___Bacterial_spot: 2297
Peach___healthy: 360
Pepper,_bell___Bacterial_spot: 997
Pepper,_bell___healthy: 1478
Potato___Early_blight: 1000
Potato___Late_blight: 1000
Potato___healthy: 152
Raspberry___healthy: 371
Soybean___healthy: 5090
Squash___Powdery_mildew: 1835
Strawberry___Leaf_scorch: 1109
Strawberry___healthy: 456
Tomato___Bacterial_spot: 2127
Tomato___Early_blight: 1000
Tomato___Late_blight: 1909

In [6]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter
import numpy as np

DATA_DIR = Path("/content/plantvillage/raw/color")
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

class_names = sorted(
    p.name for p in DATA_DIR.iterdir()
    if p.is_dir() and any(
        f.is_file() and f.suffix.lower() in EXTENSIONS
        for f in p.iterdir()
    )
)

class_to_idx = {name: i for i, name in enumerate(class_names)}

samples = [
    (str(img), class_to_idx[folder.name])
    for folder in DATA_DIR.iterdir()
    if folder.is_dir()
    for img in folder.iterdir()
    if img.is_file() and img.suffix.lower() in EXTENSIONS
]

labels = np.array([label for _, label in samples])
indices = np.arange(len(samples))

train_idx, remaining_idx = train_test_split(
    indices, test_size=0.20, random_state=42, stratify=labels
)

val_idx, test_idx = train_test_split(
    remaining_idx,
    test_size=0.50,
    random_state=42,
    stratify=labels[remaining_idx]
)

print("Total:", len(samples))
print("Training:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))
print("Classes:", len(class_names))

Total: 54305
Training: 43444
Validation: 5430
Test: 5431
Classes: 38


In [12]:
import json
import random
import numpy as np
import torch
import torch.nn as nn

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights
from sklearn.metrics import accuracy_score, classification_report

from google.colab import drive
drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected. Enable GPU in Colab.")

class LeafDataset(Dataset):
    def __init__(self, samples, indices, transform):
        self.samples = samples
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, label = self.samples[int(self.indices[i])]
        with Image.open(path) as im:
            image = im.convert("RGB")
        return self.transform(image), label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

train_ds = LeafDataset(samples, train_idx, train_transform)
val_ds = LeafDataset(samples, val_idx, eval_transform)
test_ds = LeafDataset(samples, test_idx, eval_transform)

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

weights = EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, len(class_names)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-4, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

ARTIFACT_DIR = "/content/drive/MyDrive/CropSense_AI/artifacts"
import os
os.makedirs(ARTIFACT_DIR, exist_ok=True)

best_val_acc = 0.0
best_state = None
patience = 4
stale_epochs = 0
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, targets in train_loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * targets.size(0)
        correct += (logits.argmax(1) == targets).sum().item()
        total += targets.size(0)

    model.eval()
    val_correct, val_total = 0, 0

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            logits = model(images)
            val_correct += (logits.argmax(1) == targets).sum().item()
            val_total += targets.size(0)

    train_acc = correct / total
    val_acc = val_correct / val_total
    scheduler.step(val_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"loss={total_loss/total:.4f} | "
        f"train_acc={train_acc:.4f} | "
        f"val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        torch.save(
            best_state,
            os.path.join(ARTIFACT_DIR, "plant_disease_efficientnet_b0.pth")
        )
        stale_epochs = 0
    else:
        stale_epochs += 1

    if stale_epochs >= patience:
        print("Early stopping.")
        break

if best_state is None:
    raise RuntimeError("No best model checkpoint was produced.")

model.load_state_dict(best_state)
model.eval()

with open(os.path.join(ARTIFACT_DIR, "disease_labels.json"), "w") as f:
    json.dump(class_names, f, indent=2)

print("Best validation accuracy:", best_val_acc)
print("Saved model and class labels to:", ARTIFACT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
Epoch 1/15 | loss=0.3917 | train_acc=0.9130 | val_acc=0.9904
Epoch 2/15 | loss=0.0449 | train_acc=0.9873 | val_acc=0.9937
Epoch 3/15 | loss=0.0278 | train_acc=0.9916 | val_acc=0.9954
Epoch 4/15 | loss=0.0187 | train_acc=0.9947 | val_acc=0.9958
Epoch 5/15 | loss=0.0172 | train_acc=0.9946 | val_acc=0.9976
Epoch 6/15 | loss=0.0132 | train_acc=0.9960 | val_acc=0.9954
Epoch 7/15 | loss=0.0127 | train_acc=0.9961 | val_acc=0.9980
Epoch 8/15 | loss=0.0095 | train_acc=0.9969 | val_acc=0.9971
Epoch 9/15 | loss=0.0112 | train_acc=0.9965 | val_acc=0.9971
Epoch 10/15 | loss=0.0079 | train_acc=0.9978 | val_acc=0.9978
Epoch 11/15 | loss=0.0034 | train_acc=0.9991 | val_acc=0.9980
Early stopping.
Best validation accuracy: 0.9979742173112339
Saved model and class labels to: /content/drive/MyDrive/CropSense_AI/artifacts


In [13]:
y_true, y_pred = [], []

model.eval()
with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        logits = model(images)
        predictions = logits.argmax(dim=1).cpu().numpy()

        y_pred.extend(predictions.tolist())
        y_true.extend(targets.numpy().tolist())

print("Test accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

Test accuracy: 0.9981587184680538
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       1.00      1.00      1.00        63
                                 Apple___Black_rot       1.00      1.00      1.00        62
                          Apple___Cedar_apple_rust       1.00      1.00      1.00        27
                                   Apple___healthy       1.00      1.00      1.00       165
                               Blueberry___healthy       1.00      1.00      1.00       150
          Cherry_(including_sour)___Powdery_mildew       1.00      1.00      1.00       105
                 Cherry_(including_sour)___healthy       1.00      1.00      1.00        85
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.94      0.96      0.95        51
                       Corn_(maize)___Common_rust_       1.00      1.00      1.00       120
               Corn_(maize)___Northern_Leaf_B

In [14]:

# ============================================================
# CropSense AI — Final Saved-Model Verification
# Uses the existing test_loader, model architecture and labels
# ============================================================

import os
import json
import torch
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report
from torchvision import models
import torch.nn as nn

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/CropSense_AI/artifacts"
)
MODEL_PATH = ARTIFACT_DIR / "plant_disease_efficientnet_b0.pth"
LABEL_PATH = ARTIFACT_DIR / "disease_labels.json"

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert LABEL_PATH.exists(), f"Labels not found: {LABEL_PATH}"

with open(LABEL_PATH, "r") as f:
    saved_class_names = json.load(f)

assert len(saved_class_names) == len(class_names), (
    "Saved class list length differs from training class list."
)
assert saved_class_names == list(class_names), (
    "Saved class order differs from training class order. "
    "Check how class_names was originally created."
)

# Rebuild model and load the saved best weights
verified_model = models.efficientnet_b0(weights=None)
verified_model.classifier[1] = nn.Linear(
    verified_model.classifier[1].in_features,
    len(saved_class_names)
)

state = torch.load(
    MODEL_PATH, map_location="cpu", weights_only=True
)
verified_model.load_state_dict(state)
verified_model = verified_model.to(device).eval()

# Evaluate the untouched test set
y_true, y_pred = [], []

with torch.inference_mode():
    for images, targets in test_loader:
        images = images.to(device)
        logits = verified_model(images)
        predictions = logits.argmax(dim=1).cpu().tolist()

        y_pred.extend(predictions)
        y_true.extend(targets.tolist())

test_acc = accuracy_score(y_true, y_pred)

print("=" * 55)
print("SAVED MODEL — TEST SET EVALUATION")
print("=" * 55)
print(f"Test images: {len(y_true)}")
print(f"Test accuracy: {test_acc:.4%}")
print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(saved_class_names))),
    target_names=saved_class_names,
    zero_division=0,
    digits=4
))

# Check predictions from the in-memory model against the
# reloaded model on the same deterministic test batch.
model.eval()
verified_model.eval()

probe_images, _ = next(iter(test_loader))
probe_images = probe_images.to(device)

with torch.inference_mode():
    original_predictions = model(probe_images).argmax(1)
    reloaded_predictions = verified_model(probe_images).argmax(1)

same_predictions = torch.equal(
    original_predictions, reloaded_predictions
)

print("\nReload verification:")
print("Predictions identical:", same_predictions)

if not same_predictions:
    print(
        "WARNING: The original in-memory model and saved model "
        "produce different predictions. Investigate before deployment."
    )
elif test_acc >= 0.95:
    print("Excellent test result. Continue with a real-world image audit.")
else:
    print(
        "Investigate test performance before using the model in CropSense AI."
    )


SAVED MODEL — TEST SET EVALUATION
Test images: 5431
Test accuracy: 99.8159%

Classification report:
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab     1.0000    1.0000    1.0000        63
                                 Apple___Black_rot     1.0000    1.0000    1.0000        62
                          Apple___Cedar_apple_rust     1.0000    1.0000    1.0000        27
                                   Apple___healthy     1.0000    1.0000    1.0000       165
                               Blueberry___healthy     1.0000    1.0000    1.0000       150
          Cherry_(including_sour)___Powdery_mildew     1.0000    1.0000    1.0000       105
                 Cherry_(including_sour)___healthy     1.0000    1.0000    1.0000        85
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     0.9423    0.9608    0.9515        51
                       Corn_(maize)___Common_rust_     1.0000    1.0000

In [15]:

from pathlib import Path

artifact_dir = Path(
    "/content/drive/MyDrive/CropSense_AI/artifacts"
)

files_to_check = [
    artifact_dir / "plant_disease_efficientnet_b0.pth",
    artifact_dir / "disease_labels.json",
]

for path in files_to_check:
    if path.exists():
        print(f"OK: {path.name} — {path.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"MISSING: {path}")

if all(path.exists() for path in files_to_check):
    print("\nYour trained model and labels are saved in Google Drive.")
    print("A runtime disconnect will not erase these files.")
else:
    print("\nCheck the missing files before relying on this saved model.")


OK: plant_disease_efficientnet_b0.pth — 15.75 MB
OK: disease_labels.json — 0.00 MB

Your trained model and labels are saved in Google Drive.
A runtime disconnect will not erase these files.


In [23]:

from google.colab import drive, files
from pathlib import Path

drive.mount("/content/drive")

uploaded = files.upload()

print("\nUploaded images:")
for filename in uploaded:
    print("-", filename)

if not uploaded:
    print("No images uploaded. Run this cell again to select photos.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving download (3).jpeg to download (3).jpeg

Uploaded images:
- download (3).jpeg


In [24]:

# Farmer-friendly disease prediction output

for filename, image_bytes in uploaded.items():
    import io
    from PIL import Image
    import torch

    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    x = transform(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        probabilities = torch.softmax(model(x), dim=1)[0]

    score, idx = probabilities.max(dim=0)
    predicted_label = class_names[idx.item()]

    # Split labels such as Strawberry___Leaf_scorch
    if "___" in predicted_label:
        crop_name, disease_name = predicted_label.split("___", 1)
    else:
        crop_name, disease_name = "Unknown crop", predicted_label

    crop_name = crop_name.replace("_", " ").replace(",", "")
    disease_name = disease_name.replace("_", " ").strip()

    print("\n" + "=" * 45)
    print("PLANT DISEASE DIAGNOSIS")
    print("=" * 45)
    print("Image:", filename)
    print("Predicted disease:", disease_name)
    print("Likely crop:", crop_name)
    print(f"Model score: {score.item():.2%}")

    if disease_name.lower() == "healthy":
        print("Interpretation: The model predicts a healthy leaf.")
    else:
        print("Interpretation: The model predicts a possible disease.")
        print("Confirm the result by checking symptoms and growing conditions.")

    print("=" * 45)



PLANT DISEASE DIAGNOSIS
Image: download (3).jpeg
Predicted disease: Late blight
Likely crop: Potato
Model score: 62.92%
Interpretation: The model predicts a possible disease.
Confirm the result by checking symptoms and growing conditions.


In [25]:

from google.colab import drive
drive.mount('/content/drive')

import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    dirs[:] = [d for d in dirs if d not in {
        '.ipynb_checkpoints', 'node_modules'
    }]
    if any('PlantVillage' in d for d in dirs) or \
       any('plantvillage' in f.lower() for f in files):
        print("Possible dataset location:", root)
        print("Folders:", dirs[:10])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:

import os

base = "/content/drive/MyDrive"
keywords = ["plantvillage", "plant_village", "disease"]

for root, dirs, files in os.walk(base):
    # Skip unnecessary folders
    dirs[:] = [
        d for d in dirs
        if d not in {".ipynb_checkpoints", "node_modules", ".git"}
    ]

    folder = os.path.basename(root).lower()
    matching_files = [
        f for f in files
        if f.lower().endswith(".ipynb")
        and any(k in f.lower() for k in keywords)
    ]

    if any(k in folder for k in keywords) or matching_files:
        print("\nFolder:", root)
        for f in files:
            if f.lower().endswith((".csv", ".json", ".zip", ".ipynb")):
                print("  ", f)


In [27]:

import os

DATASET_DIR = "/content/plantvillage/raw/color"

print("Dataset exists:", os.path.exists(DATASET_DIR))

if os.path.exists(DATASET_DIR):
    classes = [
        d for d in os.listdir(DATASET_DIR)
        if os.path.isdir(os.path.join(DATASET_DIR, d))
    ]

    total_images = 0

    for cls in sorted(classes):
        folder = os.path.join(DATASET_DIR, cls)
        count = sum(
            f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
            for f in os.listdir(folder)
        )
        total_images += count
        print(f"{cls}: {count}")

    print("\nTotal classes:", len(classes))
    print("Total images:", total_images)


Dataset exists: True
Apple___Apple_scab: 630
Apple___Black_rot: 621
Apple___Cedar_apple_rust: 275
Apple___healthy: 1645
Blueberry___healthy: 1502
Cherry_(including_sour)___Powdery_mildew: 1052
Cherry_(including_sour)___healthy: 854
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513
Corn_(maize)___Common_rust_: 1192
Corn_(maize)___Northern_Leaf_Blight: 985
Corn_(maize)___healthy: 1162
Grape___Black_rot: 1180
Grape___Esca_(Black_Measles): 1383
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1076
Grape___healthy: 423
Orange___Haunglongbing_(Citrus_greening): 5507
Peach___Bacterial_spot: 2297
Peach___healthy: 360
Pepper,_bell___Bacterial_spot: 997
Pepper,_bell___healthy: 1478
Potato___Early_blight: 1000
Potato___Late_blight: 1000
Potato___healthy: 152
Raspberry___healthy: 371
Soybean___healthy: 5090
Squash___Powdery_mildew: 1835
Strawberry___Leaf_scorch: 1109
Strawberry___healthy: 456
Tomato___Bacterial_spot: 2127
Tomato___Early_blight: 1000
Tomato___Late_blight: 1909
Tomato___Leaf_Mold: 

In [28]:

import os

search_roots = [
    "/content/drive/MyDrive/CropSense_AI",
    "/content/drive/MyDrive/Colab Notebooks",
    "/content"
]

keywords = [
    "train_test_split",
    "random_state",
    "test_dataset",
    "test_indices",
    "class_names"
]

for base in search_roots:
    if not os.path.exists(base):
        continue

    for root, dirs, files in os.walk(base):
        dirs[:] = [
            d for d in dirs
            if d not in {".git", "node_modules", "__pycache__"}
        ]

        for name in files:
            if name.endswith((".ipynb", ".py")):
                path = os.path.join(root, name)
                try:
                    with open(path, "r", encoding="utf-8") as f:
                        content = f.read()
                    hits = [k for k in keywords if k in content]
                    if hits:
                        print(path, "->", hits)
                except Exception:
                    pass


/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb -> ['train_test_split', 'random_state', 'class_names']
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb -> ['train_test_split', 'random_state', 'class_names']


In [29]:

import json

path = "/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb"

with open(path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(term in source for term in [
        "train_test_split",
        "random_state",
        "ImageFolder",
        "class_names",
        "test_dataset",
        "test_loader"
    ]):
        print(f"\n{'='*20} CELL {i} {'='*20}\n")
        print(source[:6000])



==================== CELL 5 ====================

from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter
import numpy as np

DATA_DIR = Path("/content/plantvillage/raw/color")
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

class_names = sorted(
    p.name for p in DATA_DIR.iterdir()
    if p.is_dir() and any(
        f.is_file() and f.suffix.lower() in EXTENSIONS
        for f in p.iterdir()
    )
)

class_to_idx = {name: i for i, name in enumerate(class_names)}

samples = [
    (str(img), class_to_idx[folder.name])
    for folder in DATA_DIR.iterdir()
    if folder.is_dir()
    for img in folder.iterdir()
    if img.is_file() and img.suffix.lower() in EXTENSIONS
]

labels = np.array([label for _, label in samples])
indices = np.arange(len(samples))

train_idx, remaining_idx = train_test_split(
    indices, test_size=0.20, random_state=42, stratify=labels
)

val_idx, test_idx = train_test_split(
    remaining_idx,
    test_size=0

In [30]:

from pathlib import Path
from sklearn.model_selection import train_test_split
import numpy as np
import json

DATA_DIR = Path("/content/plantvillage/raw/color")
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

class_names = sorted(
    p.name for p in DATA_DIR.iterdir()
    if p.is_dir() and any(
        f.is_file() and f.suffix.lower() in EXTENSIONS
        for f in p.iterdir()
    )
)

class_to_idx = {name: i for i, name in enumerate(class_names)}

samples = [
    (str(img), class_to_idx[folder.name])
    for folder in DATA_DIR.iterdir()
    if folder.is_dir()
    for img in folder.iterdir()
    if img.is_file() and img.suffix.lower() in EXTENSIONS
]

labels = np.array([label for _, label in samples])
indices = np.arange(len(samples))

train_idx, remaining_idx = train_test_split(
    indices, test_size=0.20, random_state=42, stratify=labels
)

val_idx, test_idx = train_test_split(
    remaining_idx, test_size=0.50, random_state=42,
    stratify=labels[remaining_idx]
)

with open(
    "/content/drive/MyDrive/CropSense_AI/artifacts/disease_labels.json"
) as f:
    saved_labels = json.load(f)

print("Images:", len(samples))
print("Classes:", len(class_names))
print("Train / validation / test:",
      len(train_idx), len(val_idx), len(test_idx))
print("Class mapping matches saved model:", class_names == saved_labels)
print("Example test image:", samples[int(test_idx[0])][0])


Images: 54305
Classes: 38
Train / validation / test: 43444 5430 5431
Class mapping matches saved model: True
Example test image: /content/plantvillage/raw/color/Apple___Black_rot/f15c02d9-9aa6-45b5-84fc-97e22f7fabaa___JR_FrgE.S 2831.JPG


In [31]:

import json
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report

ARTIFACT_DIR = "/content/drive/MyDrive/CropSense_AI/artifacts"
MODEL_PATH = f"{ARTIFACT_DIR}/plant_disease_efficientnet_b0.pth"
LABEL_PATH = f"{ARTIFACT_DIR}/disease_labels.json"

with open(LABEL_PATH) as f:
    saved_labels = json.load(f)

class TestDataset(Dataset):
    def __init__(self, samples, indices):
        self.samples = samples
        self.indices = indices
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                [0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, label = self.samples[int(self.indices[i])]
        with Image.open(path) as im:
            image = im.convert("RGB")
        return self.transform(image), label

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, len(saved_labels)
)
state = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(state)
model = model.to(device).eval()

test_loader = DataLoader(
    TestDataset(samples, test_idx),
    batch_size=32, shuffle=False, num_workers=2
)

y_true, y_pred = [], []

with torch.inference_mode():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        y_pred.extend(outputs.argmax(1).cpu().tolist())
        y_true.extend(labels.tolist())

print("Test images:", len(y_true))
print(f"Test accuracy: {accuracy_score(y_true, y_pred):.4%}")
print("\nPer-class precision, recall and F1:")
print(classification_report(
    y_true, y_pred,
    labels=list(range(len(saved_labels))),
    target_names=saved_labels,
    digits=4, zero_division=0
))
print("Incorrect predictions:",
      sum(a != b for a, b in zip(y_true, y_pred)))


Test images: 5431
Test accuracy: 99.8159%

Per-class precision, recall and F1:
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab     1.0000    1.0000    1.0000        63
                                 Apple___Black_rot     1.0000    1.0000    1.0000        62
                          Apple___Cedar_apple_rust     1.0000    1.0000    1.0000        27
                                   Apple___healthy     1.0000    1.0000    1.0000       165
                               Blueberry___healthy     1.0000    1.0000    1.0000       150
          Cherry_(including_sour)___Powdery_mildew     1.0000    1.0000    1.0000       105
                 Cherry_(including_sour)___healthy     1.0000    1.0000    1.0000        85
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     0.9423    0.9608    0.9515        51
                       Corn_(maize)___Common_rust_     1.0000    1.0000    1.0000       120


In [32]:

import torch
import pandas as pd
from PIL import Image
from IPython.display import display
from torchvision import models, transforms
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

ARTIFACT_DIR = "/content/drive/MyDrive/CropSense_AI/artifacts"
MODEL_PATH = f"{ARTIFACT_DIR}/plant_disease_efficientnet_b0.pth"

# Load the existing saved model; do not train it
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features, len(saved_labels)
)
state = torch.load(
    MODEL_PATH, map_location="cpu", weights_only=True
)
model.load_state_dict(state)
model = model.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

class InspectionDataset(Dataset):
    def __len__(self):
        return len(test_idx)

    def __getitem__(self, i):
        path, label = samples[int(test_idx[i])]
        with Image.open(path) as im:
            image = im.convert("RGB")
        return transform(image), label, path

loader = DataLoader(
    InspectionDataset(), batch_size=32, shuffle=False, num_workers=2
)

errors = []

with torch.inference_mode():
    offset = 0
    for images, labels, paths in loader:
        outputs = model(images.to(device))
        probs = torch.softmax(outputs, dim=1)
        scores, predictions = probs.max(dim=1)

        for j in range(len(labels)):
            true_idx = int(labels[j])
            pred_idx = int(predictions[j].cpu())

            if true_idx != pred_idx:
                errors.append({
                    "image_path": paths[j],
                    "true_label": saved_labels[true_idx],
                    "predicted_label": saved_labels[pred_idx],
                    "confidence_percent": round(
                        float(scores[j].cpu()) * 100, 2
                    )
                })
        offset += len(labels)

print("Incorrect predictions found:", len(errors))
display(pd.DataFrame(errors))


Incorrect predictions found: 10


,image_path,true_label,predicted_label,confidence_percent
0,/content/plantvillage/raw/color/Tomato___Targe...,Tomato___Target_Spot,Tomato___healthy,75.93
1,/content/plantvillage/raw/color/Corn_(maize)__...,Corn_(maize)___Northern_Leaf_Blight,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,57.97
2,/content/plantvillage/raw/color/Corn_(maize)__...,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,Corn_(maize)___Northern_Leaf_Blight,98.45
3,/content/plantvillage/raw/color/Potato___Late_...,Potato___Late_blight,Tomato___Late_blight,87.37
4,/content/plantvillage/raw/color/Tomato___Early...,Tomato___Early_blight,Tomato___Leaf_Mold,82.36
5,/content/plantvillage/raw/color/Corn_(maize)__...,Corn_(maize)___Northern_Leaf_Blight,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,53.62
6,/content/plantvillage/raw/color/Tomato___Early...,Tomato___Early_blight,Tomato___Tomato_Yellow_Leaf_Curl_Virus,77.44
7,/content/plantvillage/raw/color/Corn_(maize)__...,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,Corn_(maize)___Northern_Leaf_Blight,95.71
8,/content/plantvillage/raw/color/Potato___Late_...,Potato___Late_blight,Tomato___Late_blight,99.99
9,/content/plantvillage/raw/color/Corn_(maize)__...,Corn_(maize)___Northern_Leaf_Blight,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...,62.09
